In [80]:
from langgraph.graph import StateGraph,START,END,MessagesState 
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage 


In [81]:
from dotenv import load_dotenv 
load_dotenv()

True

In [82]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='Llama-3.3-70b-Versatile')


In [83]:
checkpointer=InMemorySaver()

In [84]:
#lets design the graph for chatting with the llm 

graph=StateGraph(MessagesState)

In [85]:
def chat_with_llm(state: MessagesState)-> MessagesState:
    
    response=llm.invoke(state['messages']) 
    
    return {'messages':[response]}

In [86]:
graph.add_node('chat',chat_with_llm)

graph.add_edge(START,'chat') 
graph.add_edge('chat',END)

In [87]:
workflow=graph.compile(checkpointer=checkpointer)

In [88]:
config={'configurable':{'thread_id':'32'}} 

In [93]:
# initial_state={'messages': [HumanMessage(content="Hi, My name is aashish")]}
initial_state={'messages':[HumanMessage(content='Do you know my name')]}

In [94]:
final_state=workflow.invoke(input=initial_state,config=config)

In [95]:
final_state

{'messages': [HumanMessage(content='Hi, My name is aashish', additional_kwargs={}, response_metadata={}, id='c7a0597f-84d0-428a-a40c-921a5a4cbc13'),
  AIMessage(content="Hello Aashish, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 43, 'total_tokens': 71, 'completion_time': 0.045781185, 'prompt_time': 0.002210813, 'queue_time': 0.055034377, 'total_time': 0.047991998}, 'model_name': 'Llama-3.3-70b-Versatile', 'system_fingerprint': 'fp_c06d5113ec', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--1b587e58-ff8f-415d-9be6-c6c4b0a819a2-0', usage_metadata={'input_tokens': 43, 'output_tokens': 28, 'total_tokens': 71}),
  HumanMessage(content='Do you know my name', additional_kwargs={}, response_metadata={}, id='22003636-6a3b-4412-80cf-20eeb86e5e19'),
  AIMessage(content='Your name is Aashish. I remember you told me that

In [ ]:
#This is simple short term memory which remains for the current execution only

In [ ]:
#if we wants that it doesn't gets deleted after the execution we have to use the dbms persistence